# FR labour-supply data walkthrough — original data, step by step

Read the **original FR microdata**, then clean / filter / price it one inspectable step at a time. You see the dataframe and the household funnel after every step. No engine-ready files, no reproduction oracles.

**What is grounded vs. what you confirm**
- The **eligibility chain** (the cleaning/filtering logic) is transcribed from your DE adapter (`dclaborsupply_app.de.data_prep`), which is a deliberate FR-mirror — so these rules *are* the FR rules.
- The **column names** are the standardized EUROMOD input variables. Cell 1 checks which are actually present in your file, so you verify them — I'm not asserting your schema.
- The **one thing I cannot know** is the path to your FR raw file. It is the single `# CONFIRM (path)` below.
- Places where **FR genuinely differs from DE** are flagged `# FR-SPECIFIC`.

## 0. Read the original FR microdata

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
pd.set_option('display.max_columns', 80); pd.set_option('display.width', 200)

# THE ONE INPUT I CANNOT KNOW. Your FR EU-SILC / EUROMOD-input microdata file.
# EUROMOD input files are tab-separated .txt (same format as the DE_2017_a2.txt the
# DE adapter reads). Fill in the real path; I am deliberately not guessing it.
FR_RAW = Path('.../FR_2016_a3.txt')   # CONFIRM (path)

raw = pd.read_csv(FR_RAW, sep='\t')   # EUROMOD input is tab-separated
print('raw shape   :', raw.shape)
print('n households:', raw['idhh'].nunique() if 'idhh' in raw else '? (no idhh col)')
print('n persons   :', len(raw))
raw.head()

## 1. Inspect the schema (you verify, not me)
Confirms which EUROMOD-standard variables your file actually carries, and which FR-specific ones are present.

In [ ]:
print('--- all columns ---'); print(sorted(raw.columns)); print()

core = ['idhh','idperson','idpartner','dag','dgn','dms','dec','ddi','deh',
        'les','lhw','loc','yem','yse','yivwg']
print('core EUROMOD vars present:', [c for c in core if c in raw.columns])
print('core EUROMOD vars MISSING:', [c for c in core if c not in raw.columns])
print()
# FR-SPECIFIC variables to look for (these are where FR differs from the DE file):
fr_specific = ['lma','drgn1','drgur','drgmd','drgru','yem00','yemxp','dwt',
               'byr','pdi','poa','psu','idfather','idmother','idorighh']
for c in fr_specific:
    print(f'  {c:10s}: {"present" if c in raw.columns else "absent"}')

## 2. Eligibility config (FR-mirror constants, from the DE adapter)
These are the frozen FR rule constants the DE adapter mirrors.

In [ ]:
CONFIG = dict(
    age_range=(18, 65),
    allowed_les=(3, 5, 7),               # employee / unemployed / inactive deciders
    wage_bounds=(2.0, 170.0),            # employee-decider yivwg bounds
    other_member_income_threshold=50.0,  # |yem|/|yse| above this = earning non-decider
    hours_cap_high=70, hours_floor_low=10, hours_inactive_threshold=5,
    retire_cols=('byr', 'pdi', 'poa', 'psu'),   # CONFIRM FR benefit-receipt columns
)
CONFIG

## 3. Classify households (single / opposite-sex couple)
Single = one adult with no partner link; couple_mf = two mutually-linked opposite-sex adults. `ruro_decider` marks the adults whose labour supply is modelled.

In [ ]:
ADULT = CONFIG['age_range'][0]
dag = pd.to_numeric(raw['dag'], errors='coerce').fillna(-1)
idp = pd.to_numeric(raw['idpartner'], errors='coerce').fillna(0).astype('int64')
id2partner = dict(zip(raw['idperson'].astype('int64'), idp))
idset = set(raw['idperson'].astype('int64'))

def _mutual(a, b):
    return b != 0 and b in idset and id2partner.get(b, 0) == a

cls = {}
for hh, g in raw.groupby('idhh'):
    ad = g[pd.to_numeric(g['dag'], errors='coerce') >= ADULT]
    n = len(ad)
    if n == 0:
        cls[hh] = 'excl_no_adult'
    elif n == 1:
        cls[hh] = 'single' if int(ad['idpartner'].iloc[0]) == 0 else 'excl_2adult_no_link'
    elif n == 2:
        a, b = ad['idperson'].astype('int64').tolist()
        if _mutual(a, b) and _mutual(b, a):
            gens = sorted(pd.to_numeric(ad['dgn'], errors='coerce').tolist())
            cls[hh] = 'couple_mf' if gens == [0, 1] else 'excl_same_sex'
        else:
            cls[hh] = 'excl_2adult_no_link'
    else:
        cls[hh] = 'excl_3plus_adults'

raw['household_class'] = raw['idhh'].map(cls)
raw['ruro_decider'] = (raw['household_class'].isin(['single', 'couple_mf'])
                       & (pd.to_numeric(raw['dag'], errors='coerce') >= ADULT)).astype(int)
print(raw.groupby('idhh')['household_class'].first().value_counts())
print('\ndeciders flagged:', int(raw['ruro_decider'].sum()))

## 4. The eligibility chain — one step per cell
Each step prints the household funnel (before -> after) so you watch the sample shrink. A tiny helper logs counts.

In [ ]:
def funnel(df, label):
    print(f'{label:48s} households={df["idhh"].nunique():6d}  persons={len(df):6d}')
    return df

def keep_all_deciders(df, cond):
    """Keep a household only if EVERY decider satisfies cond."""
    dec = df['ruro_decider'] == 1
    bad = df.loc[dec & ~cond.reindex(df.index), 'idhh']
    return df[~df['idhh'].isin(pd.unique(bad))].copy()

def drop_hh(df, bad_idhh):
    return df[~df['idhh'].isin(pd.unique(bad_idhh))].copy()

# Step 4.0 — baseline: singles + opposite-sex couples only
work = raw[raw['household_class'].isin(['single', 'couple_mf'])].copy()
funnel(work, '4.0 baseline (single + couple_mf)');

In [ ]:
# Step 4.1 — age: every decider in [18, 65]
lo, hi = CONFIG['age_range']
dag = pd.to_numeric(work['dag'], errors='coerce')
work = keep_all_deciders(work, dag.between(lo, hi))
funnel(work, '4.1 age (all deciders 18-65)');

In [ ]:
# Step 4.2 — education: every decider dec == 0 (not currently in education)
if 'dec' in work.columns:
    dec = pd.to_numeric(work['dec'], errors='coerce')
    work = keep_all_deciders(work, dec.eq(0))
funnel(work, '4.2 education (deciders dec==0)');

In [ ]:
# Step 4.3 — retirement/disability: HH sum of (byr+pdi+poa+psu) == 0
# CONFIRM these are the right FR benefit-receipt columns (DE adapter used these four).
rc = [c for c in CONFIG['retire_cols'] if c in work.columns]
print('retire cols used:', rc)
if rc:
    retire = work[rc].apply(pd.to_numeric, errors='coerce').fillna(0.0).sum(axis=1)
    work['_retire'] = retire
    hh_retire = work.groupby('idhh')['_retire'].sum()
    work = drop_hh(work, hh_retire.index[hh_retire > 0]).drop(columns='_retire')
funnel(work, '4.3 retirement/disability (HH sum == 0)');

In [ ]:
# Step 4.4 — allowed labour status: every decider les in {3, 5, 7}
les = pd.to_numeric(work['les'], errors='coerce')
work = keep_all_deciders(work, les.isin(CONFIG['allowed_les']))
funnel(work, '4.4 allowed LES (deciders in {3,5,7})');

In [ ]:
# Step 4.5 — other household members: drop HH if any NON-decider is
#   working-age-healthy-not-student  OR  earning (|yem| or |yse| > threshold)
lo, hi = CONFIG['age_range']; thr = CONFIG['other_member_income_threshold']
nondec = work['ruro_decider'] == 0
dag = pd.to_numeric(work['dag'], errors='coerce')
ddi = pd.to_numeric(work.get('ddi', 0), errors='coerce').fillna(0)
dec = pd.to_numeric(work.get('dec', 0), errors='coerce').fillna(0)
yem = pd.to_numeric(work.get('yem', 0.0), errors='coerce').fillna(0.0)
yse = pd.to_numeric(work.get('yse', 0.0), errors='coerce').fillna(0.0)
capable = dag.between(lo, hi) & ddi.eq(0) & dec.eq(0)
earning = (yem > thr) | (yse.abs() > thr)
work = drop_hh(work, work.loc[nondec & (capable | earning), 'idhh'])
funnel(work, '4.5 other members (no capable/earning non-deciders)');

In [ ]:
# Step 4.6 — hours cap + inactive transition + wage bounds (employee deciders, les==3)
cap, floor, inact = CONFIG['hours_cap_high'], CONFIG['hours_floor_low'], CONFIG['hours_inactive_threshold']
wlo, whi = CONFIG['wage_bounds']
dec_mask = work['ruro_decider'] == 1
les = pd.to_numeric(work['les'], errors='coerce')
lhw = pd.to_numeric(work['lhw'], errors='coerce').fillna(0.0)
emp = dec_mask & les.eq(3)

n_capped = int((emp & (lhw > cap)).sum())
work.loc[emp & (lhw > cap), 'lhw'] = cap                              # cap >70 to 70
lhw = pd.to_numeric(work['lhw'], errors='coerce').fillna(0.0)
work.loc[emp & (lhw > inact) & (lhw <= floor), 'lhw'] = floor         # raise (5,10] to 10
lhw = pd.to_numeric(work['lhw'], errors='coerce').fillna(0.0)

very_low = emp & (lhw <= inact)                                        # <=5h employees
become_inactive = very_low & les.isin(CONFIG['allowed_les'])
n_inactive = int(become_inactive.sum())
work.loc[become_inactive, 'lhw'] = 0
work.loc[become_inactive, 'les'] = 7
for c in ('yem', 'yse', 'yemse'):
    if c in work.columns:
        work.loc[become_inactive, c] = 0.0
work = drop_hh(work, work.loc[very_low & ~les.isin(CONFIG['allowed_les']), 'idhh'])

if 'yivwg' in work.columns:                                            # wage bounds on employees
    dec_mask = work['ruro_decider'] == 1; les = pd.to_numeric(work['les'], errors='coerce')
    yivwg = pd.to_numeric(work['yivwg'], errors='coerce')
    bad_wage = dec_mask & les.eq(3) & yivwg.notna() & ((yivwg < wlo) | (yivwg > whi))
    work = drop_hh(work, work.loc[bad_wage, 'idhh'])

print(f'capped >70h: {n_capped}   ->inactive (<=5h): {n_inactive}')
work = work.reset_index(drop=True)
funnel(work, '4.6 hours cap + wage bounds (employees)');

In [ ]:
# The surviving analytical sample, split by household type
singles = work[work['household_class'] == 'single'].reset_index(drop=True)
couples = work[work['household_class'] == 'couple_mf'].reset_index(drop=True)
print('singles households:', singles['idhh'].nunique())
print('couples households:', couples['idhh'].nunique())
work.head()

## 5. Build features (worker flag, hours bands, education, wages)
Mirrors the DE feature step. Three points are **FR-SPECIFIC** and flagged — confirm them against your MNL `enh_RURO_prep`.

In [ ]:
df = work.copy()
lhw = pd.to_numeric(df['lhw'], errors='coerce').fillna(0.0)
les = pd.to_numeric(df['les'], errors='coerce')

# FR-SPECIFIC (worker flag): DE has no lma, so DE used is_worker = (les==3)&(lhw>0).
# FR HAS lma and uses a hierarchical rule (enh_RURO_prep). Starting point below is the
# employee-and-positive-hours definition; CONFIRM whether FR layers lma on top.
if 'lma' in df.columns:
    print('NOTE: lma present -> confirm the FR hierarchical is_worker rule; using les==3 & lhw>0 for now.')
df['is_worker'] = (les.eq(3) & (lhw > 0)).astype('int8')
df['working'] = (lhw > 0).astype('int8')

# hours bands (certified RURO definitions, identical to DE)
df['working_pt1'] = ((lhw >= 18.5) & (lhw <= 20.5)).astype('int8')
df['working_pt2'] = ((lhw >= 29.5) & (lhw <= 30.5)).astype('int8')
df['working_ft']  = ((lhw >= 37.5) & (lhw <= 40.5)).astype('int8')
df['working_lh']  = ((df['working'] == 1) & (lhw >= 44.5) & (lhw <= 70.0)).astype('int8')

# education (EUROMOD deh): low {0,1,2}, mid {3,4}, high {5}
deh = pd.to_numeric(df['deh'], errors='coerce')
df['educL'] = deh.isin([0, 1, 2]).astype('int8')
df['educM'] = deh.isin([3, 4]).astype('int8')
df['educH'] = deh.eq(5).astype('int8')

# wages: realised wage 0 for non-workers; offer wage kept for everyone
yivwg = pd.to_numeric(df['yivwg'], errors='coerce').fillna(0.0)
df['wage_for_draws'] = yivwg
df['wage_ruro'] = np.where(df['is_worker'].to_numpy() == 1, yivwg.to_numpy(), 0.0)

# age_norm centred on the decider sample mean
dagn = pd.to_numeric(df['dag'], errors='coerce')
mean_age = float(dagn[df['ruro_decider'] == 1].mean())
df['age_norm'] = dagn - mean_age
df['age_norm2'] = df['age_norm'] ** 2
df['female'] = (pd.to_numeric(df['dgn'], errors='coerce') == 0).astype('int8')

# FR-SPECIFIC (region): unlike DE (constant 0, dropped), FR drgn1/drgur/drgmd/drgru VARY -> KEEP.
for c in ['drgn1', 'drgur', 'drgmd', 'drgru']:
    print(f'  region {c}: {"present (keep for FR)" if c in df.columns else "absent"}')
# FR-SPECIFIC (earnings): FR splits employment income into yem00 / yemxp (35h overtime).
#   DE used a single yem. Relevant when you mutate earnings for pricing alternatives.
print('  yem00/yemxp present:', ('yem00' in df.columns, 'yemxp' in df.columns))

df[['idhh','dag','dgn','les','lhw','is_worker','working_ft','working_lh','educL','educM','educH','wage_ruro']].head(10)

## 6. Price the observed state through EUROMOD (real connector)
Prices the cleaned **observed** rows (no alternatives, so no earnings-mutation policy needed). `EuromodConnector.run` returns the full `sim.outputs[0]`. EUROMOD wants its raw input columns, so we feed the original input fields, not the derived feature columns.

In [ ]:
from dclaborsupply_app.euromod import EuromodConnector

MODEL_ROOT = Path(r'C:\\Users\\hisham\\MNL\\EUROMOD-STORAGE\\...')   # CONFIRM (path) what em.Model() loads
FR_COUNTRY, FR_SYSTEM, FR_DATASET = 'FR', 'FR_2015', 'FR_2016_a3'    # CONFIRM system/dataset names

# Feed EUROMOD the raw input variables of the cleaned sample (drop our derived/control cols).
derived = {'household_class','ruro_decider','is_worker','working','working_pt1','working_pt2',
           'working_ft','working_lh','educL','educM','educH','wage_for_draws','wage_ruro',
           'age_norm','age_norm2','female'}
em_input = df[[c for c in df.columns if c not in derived]].copy()
print('EUROMOD input columns:', em_input.shape[1])

conn = EuromodConnector(str(MODEL_ROOT))
res = conn.run(em_input, country=FR_COUNTRY, system=FR_SYSTEM, dataset=FR_DATASET)
priced = res.output
print('priced shape:', priced.shape, ' (full output)')
print('ils_dispy present:', 'ils_dispy' in priced.columns)
if res.warnings:
    print('warnings:', res.warnings[:3])

## 7. Inspect disposable income (ils_dispy)
The consumption input to the model. Look at its distribution on the real, cleaned, priced sample.

In [ ]:
import matplotlib.pyplot as plt
c = pd.to_numeric(priced['ils_dispy'], errors='coerce')
print(c.describe())
plt.figure(figsize=(7, 4))
plt.hist(c.clip(lower=0), bins=40)
plt.title('FR disposable income (ils_dispy) - cleaned, priced sample')
plt.xlabel('EUR / month'); plt.ylabel('count'); plt.tight_layout(); plt.show()

---
## What you have after this
A clean line from your **original FR microdata** to a priced, inspectable sample, with the household funnel visible at every eligibility step. The only path I could not supply is your raw file (cell 0); the eligibility rules are your DE adapter's FR-mirror chain, the column names are EUROMOD-standard and checked in cell 1, and the FR-vs-DE differences (`is_worker`+lma, region kept, yem00/yemxp) are flagged where they bite.

**Next** (only if useful): build the latent-job alternatives from this cleaned sample (`generate_draws_long`) and price them — that step needs an FR earnings-mutation policy mirroring DE's `de_earnings_policy` (the yem00/yemxp 35h split), the one FR piece not yet written.